# Focal loss experiment (gamma=2)

First focal-loss run on the CE stack before hyperparameter sweeps.


## Colab setup

1. Place `repro/` inside your Drive project: `BirdCLEF_Project/repro/`.
2. Open any notebook from that Drive folder in Colab (right-click → Open with → Colab).
3. Runtime: **GPU** (T4 or better).
4. Colab secrets: add `KAGGLE_API_TOKEN` (key icon in the left sidebar).
5. `BirdCLEF_Project/` on Drive must also contain:
   - `perch_v2_no_dft.onnx`
   - `embeddings_v2_archive.zip` (training notebooks)
   - `embeddings_v2_TTA_archive.zip` (TTA / final model notebooks)
   - `train.csv` and `sample_submission.csv` (or download in notebook 02)

The first code cell mounts Drive and loads `birdclef` from `BirdCLEF_Project/repro/`.
Embeddings are unzipped to `/content/` for fast GPU training; checkpoints save to `repro/outputs/`.

After execution, download the notebook with outputs and re-upload for the GitHub delivery pass.


In [1]:
from google.colab import drive

import os
import sys
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/BirdCLEF_Project")
REPRO_ROOT = DRIVE_PROJECT / "repro"

if not (REPRO_ROOT / "birdclef").exists():
    raise FileNotFoundError(
        f"Expected repro at {REPRO_ROOT}. Place repro inside BirdCLEF_Project on Drive."
    )

sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)
print(f"Working directory: {REPRO_ROOT}")

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn

from birdclef.colab import colab_prepare_training

repro_root = colab_prepare_training(stage_tta=False)
print(f"Project root: {repro_root}")


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.8 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df, NUM_CLASSES, _ = prepare_baseline_data()

In [4]:
SAVE_DIR = MODELS_DIR / "focal_gamma2_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = 10
focal_fold_scores = []
best_auc = 0.0

for TRAIN_FOLD in range(5):
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")
    fold_best = 0.0

    # Fresh model/optimizer per fold so each fold is an independent CV estimate.
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        if current_auc > fold_best:
            fold_best = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    focal_fold_scores.append(fold_best)
    best_auc = max(best_auc, fold_best)

print(f"CV score: {np.mean(focal_fold_scores):.4f} (+/- {np.std(focal_fold_scores):.4f})")

100%|██████████| 7110/7110 [00:01<00:00, 3675.76it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.0262 | Val Loss: 0.6922 | Val AUC: 0.9901
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.3736 | Val Loss: 0.6868 | Val AUC: 0.9897
Epoch 03/10 | Train Loss: 0.2008 | Val Loss: 0.7193 | Val AUC: 0.9898
Epoch 04/10 | Train Loss: 0.1424 | Val Loss: 0.7562 | Val AUC: 0.9876
Epoch 05/10 | Train Loss: 0.1144 | Val Loss: 0.7657 | Val AUC: 0.9881
Epoch 06/10 | Train Loss: 0.1144 | Val Loss: 0.7708 | Val AUC: 0.9898
Epoch 07/10 | Train Loss: 0.1158 | Val Loss: 0.7722 | Val AUC: 0.9881
Epoch 08/10 | Train Loss: 0.1061 | Val Loss: 0.7668 | Val AUC: 0.9895
Epoch 09/10 | Train Loss: 0.0978 | Val Loss: 0.7739 | Val AUC: 0.9882
Epoch 10/10 | Train Loss: 0.0855 | Val Loss: 0.7773 | Val AUC: 0.9882


100%|██████████| 7110/7110 [00:01<00:00, 4149.79it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.0221 | Val Loss: 0.6905 | Val AUC: 0.9914
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.3758 | Val Loss: 0.6820 | Val AUC: 0.9921
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold1.pth
Epoch 03/10 | Train Loss: 0.2009 | Val Loss: 0.7253 | Val AUC: 0.9920
Epoch 04/10 | Train Loss: 0.1392 | Val Loss: 0.7366 | Val AUC: 0.9910
Epoch 05/10 | Train Loss: 0.1170 | Val Loss: 0.7416 | Val AUC: 0.9912
Epoch 06/10 | Train Loss: 0.1101 | Val Loss: 0.7564 | Val AUC: 0.9900
Epoch 07/10 | Train Loss: 0.1093 | Val Loss: 0.7798 | Val AUC: 0.9894
Epoch 08/10 | Train Loss: 0.1060 | Val Loss: 0.7635 | Val AUC: 0.9905
Epoch 09/10 | Train Loss: 0.0983 | Val Loss: 0.7757 | Val AUC: 0.9904
Epoch 10/10 | Train Loss: 0.0974 | Val Loss: 0.8135 | Val AUC: 0.9894


100%|██████████| 7110/7110 [00:02<00:00, 3228.53it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.0454 | Val Loss: 0.6427 | Val AUC: 0.9909
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.3808 | Val Loss: 0.6369 | Val AUC: 0.9908
Epoch 03/10 | Train Loss: 0.2087 | Val Loss: 0.6619 | Val AUC: 0.9900
Epoch 04/10 | Train Loss: 0.1440 | Val Loss: 0.6971 | Val AUC: 0.9887
Epoch 05/10 | Train Loss: 0.1218 | Val Loss: 0.7023 | Val AUC: 0.9899
Epoch 06/10 | Train Loss: 0.1149 | Val Loss: 0.7206 | Val AUC: 0.9893
Epoch 07/10 | Train Loss: 0.1111 | Val Loss: 0.7197 | Val AUC: 0.9884
Epoch 08/10 | Train Loss: 0.1061 | Val Loss: 0.7133 | Val AUC: 0.9861
Epoch 09/10 | Train Loss: 0.0964 | Val Loss: 0.7291 | Val AUC: 0.9903
Epoch 10/10 | Train Loss: 0.1000 | Val Loss: 0.7425 | Val AUC: 0.9890


100%|██████████| 7110/7110 [00:01<00:00, 4818.65it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.0411 | Val Loss: 0.6593 | Val AUC: 0.9901
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.3851 | Val Loss: 0.6407 | Val AUC: 0.9911
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold3.pth
Epoch 03/10 | Train Loss: 0.2137 | Val Loss: 0.6772 | Val AUC: 0.9911
Epoch 04/10 | Train Loss: 0.1503 | Val Loss: 0.6724 | Val AUC: 0.9914
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold3.pth
Epoch 05/10 | Train Loss: 0.1198 | Val Loss: 0.6847 | Val AUC: 0.9901
Epoch 06/10 | Train Loss: 0.1115 | Val Loss: 0.6974 | Val AUC: 0.9906
Epoch 07/10 | Train Loss: 0.1197 | Val Loss: 0.7049 | Val AUC: 0.9902
Epoch 08/10 | Train Loss: 0.1048 | Val Loss: 0.7177 | Val AUC: 0.9899
Epoch 09/10 | Train Loss: 0.098

100%|██████████| 7109/7109 [00:01<00:00, 4766.29it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 1.0457 | Val Loss: 0.6434 | Val AUC: 0.9897
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.3833 | Val Loss: 0.6490 | Val AUC: 0.9915
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/focal_gamma2_10ep/best_moe_fold4.pth
Epoch 03/10 | Train Loss: 0.2164 | Val Loss: 0.6647 | Val AUC: 0.9899
Epoch 04/10 | Train Loss: 0.1431 | Val Loss: 0.6705 | Val AUC: 0.9888
Epoch 05/10 | Train Loss: 0.1164 | Val Loss: 0.6933 | Val AUC: 0.9901
Epoch 06/10 | Train Loss: 0.1198 | Val Loss: 0.7194 | Val AUC: 0.9894
Epoch 07/10 | Train Loss: 0.1147 | Val Loss: 0.7270 | Val AUC: 0.9907
Epoch 08/10 | Train Loss: 0.1036 | Val Loss: 0.7222 | Val AUC: 0.9898
Epoch 09/10 | Train Loss: 0.0948 | Val Loss: 0.7254 | Val AUC: 0.9887
Epoch 10/10 | Train Loss: 0.1017 | Val Loss: 0.7345 | Val AUC: 0.9889
CV s